# Apply OSM-built bakery name classifier on FHRS dataset

In [ ]:
from pathlib import Path

import joblib
import pandas as pd


FHRS_DATE = "2026-07-23"

INTERIM_FOLDER = Path("../data/business/interim")
MODEL_FOLDER = Path("../models")

FHRS_PATH = (INTERIM_FOLDER/f"london_fhrs_prepared_{FHRS_DATE}.csv")

MODEL_PATH = (MODEL_FOLDER/f"osm_bakery_name_classifier.joblib")

fhrs_business_names = pd.read_csv(FHRS_PATH, usecols=["BusinessName", "BusinessNameClean"])

bakery_model = joblib.load(MODEL_PATH)

print(f"FHRS establishment rows: {len(fhrs_business_names)}")

print("Bakery name classifier loaded successfully.")

FHRS establishment rows: 81080
Bakery name classifier loaded successfully.


In [3]:
fhrs_business_names = (fhrs_business_names[
    fhrs_business_names["BusinessNameClean"].notna()
    & fhrs_business_names["BusinessNameClean"].ne("")
    ].copy()
)

fhrs_unique_names = (fhrs_business_names
    .groupby("BusinessNameClean", as_index=False)
    .agg(DisplayName=("BusinessName", "first"),
         StoreCount=("BusinessName", "size")))

print(f"FHRS rows: {len(fhrs_business_names)}")
print(f"Unique cleaned names: {len(fhrs_unique_names)}")

fhrs_unique_names.sample(10)

FHRS rows: 81080
Unique cleaned names: 65799


,BusinessNameClean,DisplayName,StoreCount
5542,beckenham sports club,Beckenham Sports Club,1
60324,tinas tea,Tinas Tea,1
49885,sheringham house,Sheringham House,1
55135,tayyabs express,Tayyabs Express,1
9503,cafe maya,Cafe Maya,1
20265,foleys,Foleys,1
61047,treatz inc,Treatz Inc.,1
42552,partnership in care albion ltd,Partnership in Care (Albion) Ltd,1
5743,bell green food and wine,Bell Green Food and Wine,1
49002,scotch bonnet grill,Scotch Bonnet Grill,1


In [4]:
fhrs_unique_names["BakeryProbability"] = (bakery_model.predict_proba(fhrs_unique_names["BusinessNameClean"])[:, 1])

fhrs_unique_names = (fhrs_unique_names.sort_values(["BakeryProbability", "StoreCount"], ascending=[False, False], ignore_index=True))

fhrs_unique_names.head(20)

,BusinessNameClean,DisplayName,StoreCount,BakeryProbability
0,bakery and cake,Bakery And Cake,1,0.999999
1,g o d cakes bakery,G.O.D Cakes / Bakery,1,0.999998
2,bakers cakes,Bakers Cakes,1,0.999997
3,cake house bakery,Cake House Bakery,5,0.999996
4,q s bakery,Q's Bakery,1,0.999995
5,b bakery,B.Bakery,1,0.999994
6,the bakery,The Bakery,2,0.999993
7,b j bakery,B.J Bakery,1,0.999992
8,east cake and bakery ltd,East Cake & Bakery Ltd,1,0.999991
9,q s bakery ltd,Q's Bakery Ltd,1,0.999989


In [5]:
candidate_results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    candidates = fhrs_unique_names[fhrs_unique_names["BakeryProbability"].ge(threshold)]

    candidate_results.append({
        "Threshold": threshold,
        "UniqueNamesToReview": len(candidates),
        "FHRSLocationsRepresented": candidates["StoreCount"].sum(),
    })

candidate_counts = pd.DataFrame(candidate_results)

candidate_counts

,Threshold,UniqueNamesToReview,FHRSLocationsRepresented
0,0.5,4957,5753
1,0.6,3936,4609
2,0.7,3241,3737
3,0.8,2657,3059
4,0.9,2112,2462


In [6]:
OUTPUT_PATH = (INTERIM_FOLDER/ f"london_fhrs_business_names_scored_{FHRS_DATE}.csv")

fhrs_unique_names.to_csv(OUTPUT_PATH,index=False,)

print(f"Scored FHRS names saved to: {OUTPUT_PATH}")

Scored FHRS names saved to: ..\data\interim\london_fhrs_business_names_scored_2026-07-23.csv


# Exploring business names in thresholds

To evaluate the effectiveness of the model and better understand the threshold level that should be chosen, some samples should be manually checked for a drop in bakery name quality and also patterns that the model is giving value to. 

In [40]:
upper_band_sample = fhrs_unique_names[fhrs_unique_names["BakeryProbability"].between(0.80, 0.90, inclusive="left")
    ].sample(n=30, random_state= 42).sort_values("BakeryProbability", ascending= False)

upper_band_sample[["DisplayName", "BusinessNameClean", "BakeryProbability",]]

,DisplayName,BusinessNameClean,BakeryProbability
2114,The Apple Blue Patisserie,the apple blue patisserie,0.899330
2118,Biscuiteers Baking Company Ltd,biscuiteers baking company ltd,0.898971
2122,Laura's Cake Kitchen,laura s cake kitchen,0.897786
2167,The Powder Patisserie,the powder patisserie,0.888901
2184,Ritchies Fresh Bake,ritchies fresh bake,0.886664
2189,Iris Sweet Treats,iris sweet treats,0.886185
2202,BAKEDATNUMBER17,bakedatnumber17,0.884654
2275,Sweetastic Ltd,sweetastic ltd,0.872287
2286,Fresh Out Of The Oven,fresh out of the oven,0.869878
2288,The Eggfree Cake Box (Plumstead),the eggfree cake box plumstead,0.869467


In [43]:
middle_band_sample = fhrs_unique_names[fhrs_unique_names["BakeryProbability"].between(0.70, 0.80, inclusive="left")
    ].sample(n=30, random_state= 42).sort_values("BakeryProbability", ascending= False)

middle_band_sample[["DisplayName", "BusinessNameClean", "BakeryProbability",]]

,DisplayName,BusinessNameClean,BakeryProbability
2659,Flour To The People,flour to the people,0.799793
2686,Glen Pat Homes Ltd,glen pat homes ltd,0.796472
2712,The Shoap,the shoap,0.791844
2720,Cupcake Muse,cupcake muse,0.790742
2727,Aamina's Sweet Treats,aamina s sweet treats,0.789953
2747,St John Bread & Wine,st john bread and wine,0.785805
2761,Pureceli Ltd,pureceli ltd,0.783784
2767,BEFORE OR AFTER LTD,before or after ltd,0.782687
2789,Dash Cakes Ltd/ Dash Desserts Ltd,dash cakes ltd dash desserts ltd,0.778364
2838,Roby Homemade Sweets,roby homemade sweets,0.769966


From the manual inspection of the samples between two bands of 0.7-0.8 and 0.8-0.9, it seems more testing and tuning is required for the classification to be upto the standard needed to create a comprehensive bakery model. The reason being that firstly, there seems to be an influence onto the classifier from common words/word-fragments such as "the", "london" and "ltd" which may have lead to many business names scoring high on BakeryProbability. From this observation alone, the next step should be a diagnostic stage to better examine the model and test whether removing words or word fragements could improve performance. Secondly, as we follow down the 0.7-0.8 threshold, the quality of bakery names seems to begin deteriorating and could be a potential spot to set the threshold, however it is important to improve the model before that decision is made.

# Checking feature weights

Before attempting to retrain the model and checking whether the word fragments that the manual inspection found where affecting the precision of bakery name identification, it is worthwhile checking if the word fragments actually held a lot of 'weight' in the logistic regression classifier and finding any other suspicious fragments that may be worth removing.

In [3]:
vectoriser = bakery_model.named_steps["tfidf"]
classifier = bakery_model.named_steps["classifier"]

feature_weight = pd.DataFrame(
    {"Feature": vectoriser.get_feature_names_out(),
     "Weight": classifier.coef_[0]}    
)

feature_weight["FeatureGapsAsDots"] = (feature_weight["Feature"].str.replace(" ", "..", regex=False))

feature_weight.sort_values("Weight", ascending= False).head(50)

,Feature,Weight,FeatureGapsAsDots
18694,bak,6.866507,bak
18698,bake,5.917738,bake
1220,bak,5.293058,..bak
14159,ake,5.196891,ake
9435,s,5.053553,..s..
20865,cake,4.884404,cake
20861,cak,4.792152,cak
1222,bake,4.735606,..bake
2004,cake,4.435262,..cake
2003,cak,4.435262,..cak


## Initial classifier limitations

By inspecting the model coefficients, we do see that there are a lot of meaningful bakery-related fragments being included. However, there are several significantly weighted fragments that seem to have no clear relevance with bakeries, which could have occured due to a number of reasons. It could be the case that the OSM-derived labels and low minimum document finding features allowed for more patterns to influence the predictions than ideal.